# MNIST 手書き数字認識 — 多層ニューラルネットワーク (MLP)

PyTorch を使って、MNIST データセットで手書き数字 (0-9) を分類する多層ニューラルネットワークを step-by-step で実装します。

## 全体の流れ
1. セットアップ
2. データの準備と可視化
3. モデル定義
4. 学習
5. 評価
6. 予測結果の可視化

## 1. セットアップ

必要なライブラリをインポートし、計算デバイス（CPU / Apple Silicon の MPS）を設定します。

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

# デバイス設定: MPS (Apple Silicon) > CUDA (NVIDIA GPU) > CPU
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print(f"使用デバイス: {device}")

## 2. データの準備

MNIST データセットは 28×28 ピクセルの手書き数字画像 (0-9) です。

- **訓練データ**: 60,000 枚
- **テストデータ**: 10,000 枚

`transforms.ToTensor()` で画像をテンソル (0〜1 の範囲) に変換し、
`transforms.Normalize()` で平均0・標準偏差1に正規化します。

In [ ]:
# ハイパーパラメータ
BATCH_SIZE = 64
LEARNING_RATE = 1e-3
EPOCHS = 10

# データの前処理: テンソル変換 + 正規化
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))  # MNIST の平均と標準偏差
])

# データセットのダウンロードとロード
train_dataset = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root="./data", train=False, download=True, transform=transform)

# DataLoader: ミニバッチ単位でデータを供給
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"訓練データ: {len(train_dataset)} 枚")
print(f"テストデータ: {len(test_dataset)} 枚")
print(f"バッチサイズ: {BATCH_SIZE}")
print(f"1エポックあたりのバッチ数: {len(train_loader)}")

### データの可視化

実際のデータがどんなものか、サンプルを表示して確認します。

In [ ]:
# 訓練データからサンプルを取得して表示
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    image, label = train_dataset[i]
    ax.imshow(image.squeeze(), cmap="gray")
    ax.set_title(f"ラベル: {label}", fontsize=12)
    ax.axis("off")
plt.suptitle("MNIST サンプル画像", fontsize=14)
plt.tight_layout()
plt.show()

## 3. モデル定義

多層ニューラルネットワーク (MLP) を定義します。

```
入力 (784) → 隠れ層1 (256) → ReLU → 隠れ層2 (128) → ReLU → 出力 (10)
```

- **入力**: 28×28 = 784 ピクセルを1次元に展開 (flatten)
- **隠れ層**: ReLU 活性化関数で非線形性を導入
- **出力**: 10クラス (数字 0-9) のスコア (logits)

※ 出力層に softmax は不要です。`CrossEntropyLoss` が内部で処理します。

### なぜ活性化関数が必要？

活性化関数がないと、各層は単なる線形変換（行列の掛け算）になる。
線形変換を何層重ねても結果は1つの線形変換と等価なので、**層を深くする意味がなくなる**。
活性化関数で非線形性を入れることで、層を重ねるほど複雑なパターンを表現できるようになる。

### なぜ ReLU？ — 勾配消失問題との関係

#### 誤差逆伝播と活性化関数の微分

学習時、`∂Loss/∂W`（損失の重みに対する勾配）を連鎖律で計算する。
順伝播で `z → σ(z) → 次の層` と活性化関数を通しているため、
逆伝播で戻るときも同じ道を通り、**活性化関数の微分 σ'(z) が掛かる**。

3層の例（W₁ の勾配を求める場合）:
```
∂Loss/∂W₁ = ∂L/∂z₃ × W₃ × σ'(z₂) × W₂ × σ'(z₁) × x
                            ~~~~~~~        ~~~~~~~
                            通り道にある活性化関数の傾きが掛かる
```

σ は調整対象ではないが、**順伝播で信号が通る道にいるため、その傾き（微分値）が勾配の伝達効率を決める**。

#### Sigmoid の問題

Sigmoid の微分値は最大でも **0.25**。層を通るたびに勾配が最大 1/4 に減衰する。

| 層数 | σ' の回数 | 勾配の最大値 |
|------|----------|-------------|
| 3層  | 2回      | 0.25² = 0.0625 |
| 5層  | 4回      | 0.25⁴ ≈ 0.004 |
| 10層 | 9回      | 0.25⁹ ≈ 0.000004 |

→ 入力に近い層ほど勾配が消え、学習が進まなくなる（**勾配消失問題**）。

#### ReLU の解決策

ReLU: `f(x) = max(0, x)` は `x > 0` のとき微分が **常に1**。
掛け算しても減衰しないので、深い層でも勾配が消えにくい。
ただし `x ≤ 0` で微分が 0 になりニューロンが死ぬ（dying ReLU）副作用がある。

In [ ]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.layers = nn.Sequential(
            nn.Linear(28 * 28, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 10),
        )

    def forward(self, x):
        x = self.flatten(x)    # (batch, 1, 28, 28) → (batch, 784)
        x = self.layers(x)     # (batch, 784) → (batch, 10)
        return x

model = MLP().to(device)
print(model)

# パラメータ数を確認
total_params = sum(p.numel() for p in model.parameters())
print(f"\n総パラメータ数: {total_params:,}")

## 4. 学習の準備

- **損失関数**: `CrossEntropyLoss` — 多クラス分類の標準的な損失関数
- **最適化アルゴリズム**: `Adam` — 学習率を自動調整してくれる優秀なオプティマイザ

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

print(f"損失関数: {criterion}")
print(f"最適化: Adam (lr={LEARNING_RATE})")
print(f"エポック数: {EPOCHS}")

## 5. 学習ループ

各エポックで以下を繰り返します:

1. ミニバッチを取得
2. モデルに入力して予測を得る (forward)
3. 損失を計算
4. 勾配を計算 (backward)
5. パラメータを更新 (optimizer.step)
6. 勾配をリセット (optimizer.zero_grad)

In [ ]:
train_losses = []

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0

    for batch_idx, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device), labels.to(device)

        # Forward: 予測を計算
        outputs = model(images)
        loss = criterion(outputs, labels)

        # Backward: 勾配を計算してパラメータを更新
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    # エポックごとの平均損失
    avg_loss = running_loss / len(train_loader)
    train_losses.append(avg_loss)
    print(f"Epoch [{epoch+1}/{EPOCHS}]  Loss: {avg_loss:.4f}")

print("\n学習完了!")

## 6. 評価

テストデータ (学習に使っていないデータ) でモデルの性能を評価します。

In [ ]:
model.eval()
correct = 0
total = 0

with torch.no_grad():  # 評価時は勾配計算不要
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, dim=1)  # 最大スコアのクラスを予測とする
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total
print(f"テストデータでの正解率: {accuracy:.2f}% ({correct}/{total})")

### 学習曲線

エポックごとの損失 (loss) の推移をプロットします。損失が減少していれば学習が進んでいる証拠です。

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(range(1, EPOCHS + 1), train_losses, marker="o")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("学習曲線 (Training Loss)")
plt.grid(True)
plt.show()

## 7. 予測結果の可視化

テスト画像に対するモデルの予測を表示します。
- **緑**: 正解
- **赤**: 不正解

In [ ]:
model.eval()
fig, axes = plt.subplots(3, 5, figsize=(14, 9))

with torch.no_grad():
    for i, ax in enumerate(axes.flat):
        image, label = test_dataset[i]
        output = model(image.unsqueeze(0).to(device))
        pred = output.argmax(dim=1).item()

        ax.imshow(image.squeeze(), cmap="gray")
        color = "green" if pred == label else "red"
        ax.set_title(f"予測: {pred} (正解: {label})", color=color, fontsize=11)
        ax.axis("off")

plt.suptitle("テストデータに対する予測結果", fontsize=14)
plt.tight_layout()
plt.show()